# `FULL_EPISODE_UPDATES`: actor/critic impact + formula change

The training loops always decide **when** to call `agent.update()`; the agents then build their targets/advantages from the provided trajectory **and the `dones` flags**. When `FULL_EPISODE_UPDATES=False`, the last transition in the partial trajectory typically has `done=0`, so targets **bootstrap** using the critic; when `FULL_EPISODE_UPDATES=True`, the final transition is terminal (`done=1`), so bootstrapping is **cut off**.

Below, “affects” means the loss depends on the changed targets/advantages/Q-values.

---

## AC (`ac_config["FULL_EPISODE_UPDATES"] = [True]`)
- **Applies to:** **both actor and critic**
- **Short explanation + formula delta**
  - In `TSP_AC_Agent.update()`:
    - **Full-episode mode (`True`)** builds Monte-Carlo returns
      \(G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k\) with **no extra tail bootstrap** when the episode ends (`done[-1]=1` → `bootstrap=0`).
    - **Step-based mode (`False`)** bootstraps the open tail if the last transition is non-terminal (`done[-1]=0`):
      - `bootstrap = V(s_{T+1})`
      - \(G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k + \gamma^{(T-t)+1} V(s_{T+1})\)
  - **Actor loss:** `actor_loss = - Σ_t (G_t * logπ(a_t|s_t))`
  - **Critic loss:** `critic_loss = Σ_t (G_t - V(s_t))^2`
  - Because both losses use the same \(G_t\), toggling `FULL_EPISODE_UPDATES` changes **both**.

---

## A2C (`a2c_config["FULL_EPISODE_UPDATES"] = [True, False]`)
- **Applies to:** **both actor and critic**
- **Short explanation + formula delta**
  - In `A2C_Agent`, the learning target is always an n-step bootstrapped return estimate:
    \(\hat{q}_t = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})\) **unless** an episode terminal appears inside the n-step window.
  - The effect of `FULL_EPISODE_UPDATES` comes from the trajectory boundary and `dones`:
    - **Full-episode (`True`)**: the trajectory ends with `done=1`, so bootstrapping beyond the real end is suppressed.
    - **Step-based (`False`)**: the partial trajectory usually ends with `done=0` (non-terminal tail), so \(\gamma^n V(\cdot)\) bootstrapping appears more often in \(\hat{q}_t\).
  - **Actor loss:** `actor_loss = - Σ_t (A_t * logπ(a_t|s_t))`, where `A_t = \hat{q}_t - V(s_t)`
  - **Critic loss:** `critic_loss = Σ_t (\hat{q}_t - V(s_t))^2`
  - Since both use \(\hat{q}_t\) (and thus depend on whether the tail is treated as terminal), toggling affects **both**.

---

## SAC (`sac_config["FULL_EPISODE_UPDATES"] = [True]`)
- **Applies to:** **both actor and critic** (critic directly; actor via updated Q-values)
- **Short explanation + formula delta**
  - In `TSP_SAC_Agent.update()` the critic TD target explicitly uses `dones`:
    \(\text{td target}_t = r_t + \gamma(1-done_t)\, v_{t+1}\)
    where \(v_{t+1}\) is the soft value computed from the twin target critics.
  - **Full-episode (`True`)**: last step has `done=1` → \((1-done)=0\) → **no bootstrap** past the end.
  - **Step-based (`False`)**: last step typically has `done=0` → \((1-done)=1\) → **bootstrap** with target Q/soft value at the tail.
  - **Critic losses:** `((Q(s_t,a_t) - td_target_t)^2).sum()` for both critics.
  - **Actor loss:** `actor_loss = Σ_t Σ_a π(a|s_t) * (α logπ(a|s_t) - min(Q1,Q2))`
    - `done` does not appear directly in the actor loss, but **Q-values come from the critic networks**, which were trained with the bootstrapped `td_target`.
  - Therefore, toggling `FULL_EPISODE_UPDATES` changes **both** networks’ learning signal.

---

## PPO (`ppo_config["FULL_EPISODE_UPDATES"] = [True]`)
- **Applies to:** **both actor and critic**
- **Short explanation + formula delta**
  - PPO uses GAE with `done` in the TD residual:
    - \(\delta_t = r_t + \gamma(1-done_t)V(s_{t+1}) - V(s_t)\)
    - \(A_t = \delta_t + \gamma\lambda(1-done_t)A_{t+1}\)
    - returns: \(\text{returns} = A_t + V(s_t)\)
  - **Full-episode (`True`)**: the final step is terminal (`done=1`) → \((1-done)=0\), so advantages/returns stop bootstrapping at the true end.
  - **Step-based (`False`)**: the partial tail usually has `done=0` → \((1-done)=1\), so advantages/returns include bootstrapped \(V(s_{t+1})\) for the open tail.
  - **Actor loss (clipped surrogate):** depends on \(A_t\)
  - **Critic loss:** `critic_loss = value_coef * mean((V(s_t) - returns_t)^2)` depends on \(returns_t\)
  - So the change in bootstrapping alters both actor and critic objectives.
